# Exercise 4.4 — Is a predicted 0.7 worth 0.7?

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-04-calibration.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

> **A suggested addition, not part of the original module**
>
> This exercise has no page in the original version of the module. It answers a
> question Exercise 4.3 leaves open: the threshold there assumed that the
> predicted probabilities mean what they say. Accept, rewrite or remove it;
> nothing else depends on it.

Exercise 4.3 chose a threshold of 0.2 because of what each mistake costs. That
choice assumed something it never checked: that among the people the model
gives a probability of about 0.2, about 20% really do have diabetes. A model
whose probabilities match what happens in this way is **calibrated**.

Accuracy, sensitivity and specificity cannot tell you whether a model is
calibrated, because they throw the probability away at the threshold and only
look at the predicted classes.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

The same people, split and model as Exercise 4.3: `diabetes ~ glucose`, fitted
on 600 people, with predicted probabilities for both the training rows and the
150 test patients.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["glucose", "mass", "age", "diabetes"]]
diabetes["diabetes_dummy"] = (diabetes["diabetes"] == "pos").astype(int) # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

log_fit = LogisticRegression(C=np.inf)
log_fit = log_fit.fit(train[["glucose"]], train["diabetes_dummy"])

train_preds = train.copy()
train_preds["prob"] = log_fit.predict_proba(train[["glucose"]])[:, 1]
test_preds = test.copy()
test_preds["prob"] = log_fit.predict_proba(test[["glucose"]])[:, 1]

------------------------------------------------------------------------

### Part 1 — Calibration in the large

The simplest check: is the **average** predicted probability equal to the
proportion of people who actually have diabetes (the **prevalence**)? Do it on
the training rows and on the test patients:

In [ ]:
print("training rows: mean predicted", round(train_preds["prob"].mean(), 4),
      "| prevalence", round(train_preds["diabetes_dummy"].mean(), 4))
print("test patients: mean predicted", round(test_preds["prob"].mean(), 4),
      "| prevalence", round(test_preds["diabetes_dummy"].mean(), 4))

On the training rows the two numbers are identical, 0.367. That is not evidence
of anything: a logistic regression with an intercept always makes its average
fitted probability equal the prevalence of the rows it was fitted on. On the
test patients the model predicts 0.349 on average, and 0.353 of them have
diabetes: close.

This is the weakest calibration check there is, one number against one number.
A model that gave **everybody** a probability of 0.353 would pass it perfectly,
and be useless.

> **Learning points**
>
> - A model is calibrated when, among the people it gives a probability $q$, a proportion of about $q$ have the outcome.
> - Calibration in the large compares the mean predicted probability with the prevalence. Check it on the test set: on the training rows a logistic regression passes by construction.

------------------------------------------------------------------------

### Part 2 — The calibration table, step by step

To look closer, put the test patients into five groups (**bins**) by their
predicted probability: 0 to 0.2, 0.2 to 0.4, and so on. In each bin count the
people, average their predicted probabilities, and compute the proportion who
actually have diabetes. For a calibrated model the last two numbers are close in
every bin.

Step 1: give each patient a bin.

In [ ]:
test_binned = test_preds.copy()
test_binned["bin"] = pd.cut(test_binned["prob"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
                            include_lowest=True)

test_binned[["glucose", "diabetes_dummy", "prob", "bin"]].head()

Step 2: summarise each bin. Fill in the observed rate, the proportion of people
in the bin who have diabetes:

In [ ]:
# Replace every ______ below, then run this cell.
calibration_table = (test_binned
    .groupby("bin", observed=False)
    .agg(n=("prob", "size"),
         mean_predicted=("prob", "mean"),
         observed_rate=("______", "mean"))
    .reset_index())

calibration_table.round(3)

> **Stuck?**
>
> `diabetes_dummy` is 1 for diabetes and 0 for none, so its mean in a bin is the
> proportion of the bin with diabetes.

The **calibration curve** draws this table: mean predicted against observed
rate, one point per bin, sized by the number of patients in it. A perfectly
calibrated model puts every point on the dashed diagonal.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.plot(calibration_table["mean_predicted"], calibration_table["observed_rate"],
        color="steelblue")
ax.scatter(calibration_table["mean_predicted"], calibration_table["observed_rate"],
           s=calibration_table["n"] * 5, color="steelblue")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability in the bin")
ax.set_ylabel("Observed rate in the bin")
plt.show()

The middle of the range is close to the diagonal. In the lowest bin the model
says 14% on average, and 3 of the 44 patients (7%) have diabetes.

**Is a gap a fault in the model, or just a small bin?** The top bin holds 6
patients: one more or one fewer person with diabetes moves its observed rate by
0.17. A gap in a bin of 6 people tells you almost nothing; a gap in a bin of 60
tells you much more. This is why a calibration table should always show the
number of patients in each bin, and why you should not read a story into every
wobble of the curve.

> **Learning points**
>
> - The calibration table bins the predicted probabilities and compares the mean prediction with the observed rate in each bin.
> - On the calibration curve, points on the diagonal mean the probabilities can be taken at face value.
> - Small bins give noisy observed rates: always report the number of patients per bin.

------------------------------------------------------------------------

### Part 3 — Same order, different numbers

Now break the probabilities on purpose. Squash every prediction two thirds of
the way towards 0.5:

$$q = 0.5 + \frac{p - 0.5}{3}$$

A probability of 0.05 becomes 0.35, 0.5 stays 0.5, and 0.86 becomes 0.62. The
patients come out **in exactly the same order**: whoever had the higher
probability before still has the higher probability after. Only the numbers
change.

In [ ]:
test_preds["squashed"] = 0.5 + (test_preds["prob"] - 0.5) / 3

print("before:", round(test_preds["prob"].min(), 3), "to", round(test_preds["prob"].max(), 3))
print("after: ", round(test_preds["squashed"].min(), 3), "to", round(test_preds["squashed"].max(), 3))
print("same order:", (test_preds["prob"].rank() == test_preds["squashed"].rank()).all())

**The Brier score** measures how far the probabilities are from what happened.
For each patient take the squared difference between the predicted probability
and the outcome coded 0 or 1, then average:

$$\text{Brier} = \frac{1}{m}\sum_{i=1}^{m} (\hat{p}_i - y_i)^2$$

It is the mean squared error from Exercise 3.2, applied to probabilities: 0 is
perfect, and lower is better. Compute it before and after squashing:

In [ ]:
# Replace every ______ below, then run this cell.
brier_before = ((test_preds["prob"] - test_preds["diabetes_dummy"]) ** 2).mean()
brier_after  = ((______ - test_preds["diabetes_dummy"]) ** 2).mean()

print("Brier before:", round(brier_before, 3))
print("Brier after: ", round(brier_after, 3))

**Why it matters.** Apply the threshold from Exercise 4.3, 0.2, to both sets of
probabilities, and count the people sent for a test and the cost (5 for a missed
case, 1 for an unnecessary test). Then do the same at 0.5:

In [ ]:
truth = test_preds["diabetes_dummy"]
for threshold in [0.2, 0.5]:
    for name in ["prob", "squashed"]:
        p = test_preds[name]
        tested = (p >= threshold).sum()
        cost = 5 * ((truth == 1) & (p < threshold)).sum() + ((truth == 0) & (p >= threshold)).sum()
        label = "before" if name == "prob" else "after "
        print(f"threshold {threshold}, {label}: {tested} tested, cost {cost}")

At 0.2 the original probabilities send 106 of the 150 patients for a test, at a
cost of 71. The squashed probabilities send **all 150**, at a cost of 97:
exactly what testing everybody without any model would cost. Everything the
threshold gained has been lost, although the patients are in the same order as
before.

At 0.5 the two are identical, 31 tested at a cost of 152, because 0.5 is the one
value the squashing leaves where it was. A team that never moved off the default
threshold would never notice the problem. Calibration matters the moment a
probability is compared with a number: a threshold chosen from costs, or a risk
told to a patient.

> **Learning points**
>
> - Two models can put patients in exactly the same order and still give different probabilities.
> - The Brier score, the mean squared error of the probabilities, notices when the probabilities are wrong.
> - A threshold chosen from costs is only right if the probabilities are calibrated.

## Stretch

> **Stretch — balance the classes, and watch the probabilities move**
>
> 36.7% of the training rows have diabetes. Keep all of them, draw the same
> number of people without diabetes at random, and refit `diabetes ~ glucose` on
> this balanced training set. Predict the same 150 test patients. What happens to
> the mean predicted probability, to the calibration table, and to the cost at
> the threshold of 0.2? Does the order of the patients change?